# Persuasion Index Analysis Example

This notebook shows the standard ways to run the Persuasion Index scorer for a single argument, a DataFrame of arguments, seeded lexicons, and the UKP logistic-regression weighting profile.


## 1. Setup

Run this cell first. The convenience API uses the expanded lexicons by default. Use `lexicon="seeded"` or `use_expanded_lexicons=False` only when you want the original seed lexicons.


In [1]:
import pandas as pd

from persuasion_runner import build_score_matrices, score_persuasion
from persuasion_profile import get_persuasion_report


## 2. Score One Argument

For a single string, `score_persuasion` returns the raw nested category dictionary by default. Each category includes sub-feature scores and a `mean`.


In [2]:
text = "Sample text to analyze for persuasion features."

# Default behavior: expanded lexicons + raw nested score dictionary.
single_scores = score_persuasion(text)
single_scores


spaCy model unavailable (en_core_web_sm). NER-based features will be 0.0. Error: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.
VADER unavailable. vader_compound will be 0.0. Error: No module named 'vaderSentiment'


Loading lexicons from: /Users/lianchenggong/Documents/GitHub/Persuasion_Index_Code/helper_features/lexicons_expanded_LLM_audited.json


{'Evidence': {'statistical': 0.0,
  'attribution': 0.0,
  'named_entities': 0.0,
  'mean': 0.0},
 'Specificity': {'psychological_nearness': 0.0,
  'lexical_concreteness': 0.4208333333333333,
  'interactional_immediacy': 0.0,
  'mean': 0.14027777777777775},
 'Authority/Credibility': {'titles': 0.0,
  'organizations': 0.0,
  'phrases': 0.0,
  'consensus': 0.0,
  'speech_power': 1.0,
  'mean': 0.2},
 'Logic/Cohesion': {'structural_reasoning': 0.0,
  'discourse_cohesion': 0.0,
  'mean': 0.0},
 'Argumentation': {'conclusion_explicitness': 0.0,
  'premise_density': 1.0,
  'quantity_intensity': 1.0,
  'style_sophistication': 0.0,
  'mean': 0.5},
 'Opponent’s View': {'acknowledge': 0.0,
  'refutation_strength': 0.0,
  'mean': 0.0},
 'Sentiment': {'vader_compound': 0.0,
  'language_intensity': 0.0,
  'fear_threat': 0.0,
  'joy_gain': 0.0,
  'anger': 0.0,
  'sadness': 0.0,
  'valence': 0.5815714285714286,
  'arousal': 0.44342857142857145,
  'dominance': 0.5187142857142857,
  'mean': 0.1715238095

If you want DataFrame output for one text, set `output="matrices"`. This returns the same two matrices used for batch analysis.


In [3]:
# One-row matrices for a single argument.
single_sub, single_mean = score_persuasion(text, output="matrices")
single_mean


Loading lexicons from: /Users/lianchenggong/Documents/GitHub/Persuasion_Index_Code/helper_features/lexicons_expanded_LLM_audited.json


,Evidence.mean,Specificity.mean,Authority/Credibility.mean,Logic/Cohesion.mean,Argumentation.mean,Opponent’s View.mean,Sentiment.mean,Politeness.mean,Reciprocity.mean,Impact.mean,Commitment.mean,Scarcity/Urgency.mean,Engagement.mean,Propaganda.mean,Style.mean
0,0.0,0.140278,0.2,0.0,0.5,0.0,0.171524,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.437963


## 3. Score a DataFrame

For a DataFrame, provide the text column name. The function returns two DataFrames: one with all sub-features and one with category means.


In [4]:
example_df = pd.DataFrame({
    "argument": [
        "I think this product is amazing and will change your life!",
        "This is the worst experience I've ever had. Do not buy this.",
        "It's okay, not great but not terrible either.",
        "I have no strong feelings about this one way or the other.",
    ]
})

example_df


,argument
0,I think this product is amazing and will chang...
1,This is the worst experience I've ever had. Do...
2,"It's okay, not great but not terrible either."
3,I have no strong feelings about this one way o...


In [5]:
# DataFrame input defaults to expanded lexicons and returns matrices.
X_sub, X_mean = score_persuasion(example_df, text_col="argument")


Loading lexicons from: /Users/lianchenggong/Documents/GitHub/Persuasion_Index_Code/helper_features/lexicons_expanded_LLM_audited.json


In [6]:
# Every individual persuasion sub-feature, such as Evidence.statistical.
X_sub


,Evidence.statistical,Evidence.attribution,Evidence.named_entities,Specificity.psychological_nearness,Specificity.lexical_concreteness,Specificity.interactional_immediacy,Authority/Credibility.titles,Authority/Credibility.organizations,Authority/Credibility.phrases,Authority/Credibility.consensus,...,Engagement.inquiry,Engagement.past,Engagement.imagery,Engagement.characters,Propaganda.emotional_charge,Propaganda.logical_distortion,Propaganda.heuristic_identity_appeals,Style.fluency,Style.length,Style.rhetorical_punctuation
0,0.0,0.0,0.0,1.0,0.384091,1.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.999760,0.0,0.0,0.0,1.0,0.366421,0.328165
1,0.0,0.0,0.0,1.0,0.245000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.956063,0.0,0.956063,0.0,0.0,0.0,1.0,0.404745,0.318688
2,0.0,0.0,0.0,0.0,0.241071,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,1.0,0.0,0.0,1.0,0.366421,0.328165
3,0.0,0.0,0.0,1.0,0.366136,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.978638,0.0,0.0,0.0,1.0,0.377008,0.000000


In [7]:
# Category-level means, such as Evidence.mean.
X_mean


,Evidence.mean,Specificity.mean,Authority/Credibility.mean,Logic/Cohesion.mean,Argumentation.mean,Opponent’s View.mean,Sentiment.mean,Politeness.mean,Reciprocity.mean,Impact.mean,Commitment.mean,Scarcity/Urgency.mean,Engagement.mean,Propaganda.mean,Style.mean
0,0.0,0.794697,0.003101,0.492248,0.496124,0.0,0.423527,0.25,0.000000,0.25,0.0,0.492248,0.494792,0.000000,0.564862
1,0.0,0.415000,0.200000,0.977066,0.000000,0.0,0.293485,0.25,0.000000,0.00,0.0,0.499035,0.478032,0.000000,0.574477
2,0.0,0.080357,0.003101,0.984496,0.119878,0.0,0.505006,0.25,0.333333,0.25,0.0,0.000000,0.000000,0.333333,0.564862
3,0.0,0.455379,0.004272,0.489319,0.000000,0.0,0.388544,0.25,0.000000,0.00,0.0,0.489319,0.326213,0.000000,0.459003


## 4. Use Seeded Lexicons Instead

The expanded lexicons are recommended as the default. Use seeded/original lexicons when you need to reproduce seed-only results or compare against the expanded version.


In [8]:
# Option A: through the main convenience function.
seeded_sub, seeded_mean = score_persuasion(
    example_df,
    text_col="argument",
    lexicon="seeded",
)

seeded_mean


Loading lexicons from: /Users/lianchenggong/Documents/GitHub/Persuasion_Index_Code/helper_features/lexicons.json


,Evidence.mean,Specificity.mean,Authority/Credibility.mean,Logic/Cohesion.mean,Argumentation.mean,Opponent’s View.mean,Sentiment.mean,Politeness.mean,Reciprocity.mean,Impact.mean,Commitment.mean,Scarcity/Urgency.mean,Engagement.mean,Propaganda.mean,Style.mean
0,0.0,0.794697,0.003101,0.492248,0.496124,0.0,0.421805,0.25,0.0,0.25,0.0,0.492248,0.494792,0.000000,0.564862
1,0.0,0.415000,0.200000,0.977066,0.000000,0.0,0.293485,0.25,0.0,0.00,0.0,0.499035,0.478032,0.000000,0.574477
2,0.0,0.080357,0.200000,0.984496,0.119878,0.0,0.501042,0.00,0.0,0.25,0.0,0.000000,0.000000,0.333333,0.564862
3,0.0,0.455379,0.200000,0.489319,0.000000,0.0,0.277484,0.00,0.0,0.00,0.0,0.489319,0.326213,0.000000,0.459003


In [9]:
# Option B: through the matrix-specific helper.
seeded_sub_2, seeded_mean_2 = build_score_matrices(
    example_df,
    text_col="argument",
    use_expanded_lexicons=False,
)


Loading lexicons from: /Users/lianchenggong/Documents/GitHub/Persuasion_Index_Code/helper_features/lexicons.json


## 5. UKP Logistic-Regression Persuasion Profile

`get_persuasion_report` applies stored UKP logistic-regression weights to the raw scores. It returns `(raw_scores, ukp_weighted)`. The weighted result includes two probability estimates: one from sub-feature weights and one from category-mean weights.


In [10]:
profile_text = example_df.loc[0, "argument"]

# Defaults to expanded lexicons. Set use_expanded_lexicons=False for seeded lexicons.
raw_scores, ukp_weighted = get_persuasion_report(
    profile_text,
    use_expanded_lexicons=True,
)

if ukp_weighted is None:
    raise FileNotFoundError(
        "UKP coefficient files were not found in helper_features/regression_outputs."
    )

ukp_weighted["metadata"]


Loading lexicons from: /Users/lianchenggong/Documents/GitHub/Persuasion_Index_Code/helper_features/lexicons_expanded_LLM_audited.json


{'sub_model': {'log_odds': 0.4794, 'prob': np.float64(0.6176)},
 'mean_model': {'log_odds': 0.3959, 'prob': np.float64(0.5977)}}

In [11]:
# Present the final logistic-regression outputs in a compact table.
ukp_summary = pd.DataFrame(ukp_weighted["metadata"]).T
ukp_summary


,log_odds,prob
sub_model,0.4794,0.6176
mean_model,0.3959,0.5977


In [12]:
# Optional: inspect the weighted contribution from one category.
ukp_weighted["Evidence"]


{'statistical': 0.0, 'attribution': 0.0, 'named_entities': 0.0, 'mean': 0.0}